# 08 — Syntax, Constituency & Dependency Representations

**Learning objective.** Represent grammatical structure and inspect head–dependent relationships without requiring a downloaded parser model.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


**Constituency** groups tokens into nested phrases (NP, VP…). **Dependency parsing** assigns a syntactic head and relation to each token. The latter is often convenient for relation extraction because predicates and arguments become explicit edges.

In [2]:
from nltk import RegexpParser
sentence=[('The','DT'),('researcher','NN'),('trained','VBD'),('a','DT'),('model','NN'),('successfully','RB')]
grammar=r'''NP: {<DT>?<JJ.*>*<NN.*>+}
            VP: {<VB.*><NP>?<RB.*>*}'''
tree=RegexpParser(grammar).parse(sentence)
print(tree)

(S
  (NP The/DT researcher/NN)
  (VP trained/VBD (NP a/DT model/NN) successfully/RB))


In [3]:
# Encode a dependency graph explicitly using spaCy's Doc data structure.
from spacy.tokens import Doc
from spacy.vocab import Vocab
words=['The','researcher','trained','a','model','successfully']
# absolute head token indices: The→researcher, researcher→trained, trained→trained(root), a→model, model→trained, successfully→trained
heads=[1,2,2,4,2,2]
deps=['det','nsubj','ROOT','det','dobj','advmod']
doc=Doc(Vocab(), words=words, heads=heads, deps=deps)
for t in doc:
    print(f'{t.text:12s} --{t.dep_:6s}--> {t.head.text}')

The          --det   --> researcher
researcher   --nsubj --> trained
trained      --ROOT  --> trained
a            --det   --> model
model        --dobj  --> trained
successfully --advmod--> trained


---
    ## Production takeaways
    - Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
    - Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
    - Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Contrast constituency and dependency structure
- Read a dependency edge as head + relation + dependent